# Apprentissage Automatique : Région Nouvelle-Aquitaine
Modèles de prédiction des résultats électoraux en fonction des indicateurs socio-économiques.
Ce notebook suit la même logique que `Apprentissage_binaire_2017.ipynb` mais sur les données propres à la Nouvelle-Aquitaine.

In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import os
import pickle
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ============================================================================
# 1. CHARGEMENT & NETTOYAGE DES DONNÉES
# ============================================================================
data_file = r'c:\Users\tarek\Downloads\aliMSPR\MSPR_Final\MSPR\01_Donnees\data_nouvelle_aquitaine_final.csv'

# Utilisation de engine='python' sans low_memory
df = pd.read_csv(data_file, engine='python', quotechar='"', sep=',')

print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# Si les colonnes Nom ne sont pas trouvées, on tente un split manuel sur la première colonne
if 'Nom' not in df.columns:
    first_col = df.columns[0]
    if ',' in first_col:
        cols_split = df[first_col].str.split(',', expand=True)
        header_parts = first_col.replace('"', '').split(',')
        n_cols = cols_split.shape[1]
        cols_split.columns = [header_parts[i] if i < len(header_parts) else f"Col_{i}" for i in range(n_cols)]
        df = pd.concat([cols_split, df.drop(columns=[first_col])], axis=1)

def get_winner(row):
    max_v = -1
    winner = "INCONNU"
    for i in range(10):
        suffix = f".{i}" if i > 0 else ""
        nom_col = f"Nom{suffix}"
        voix_col = f"% Voix/Exp{suffix}"
        
        if nom_col in df.columns and voix_col in df.columns:
            try:
                v_val = row[voix_col]
                if pd.isna(v_val): continue
                v = float(str(v_val).replace(',', '.'))
                if v > max_v:
                    max_v = v
                    winner = str(row[nom_col]).upper().strip()
            except:
                pass
    return winner

print("Étape 1 : Extraction du vainqueur...")
df['Vainqueur'] = df.apply(get_winner, axis=1)

mapping_bord = {
    'LE PEN': 'ExD', 'ZEMMOUR': 'ExD', 'DE VILLIERS': 'ExD',
    'FILLON': 'D', 'PÉCRESSE': 'D', 'PECRESSE': 'D', 'DUPONT-AIGNAN': 'D', 'SARKOZY': 'D',
    'MACRON': 'Centre', 'LASSALLE': 'Centre', 'BAYROU': 'Centre',
    'HAMON': 'G', 'HIDALGO': 'G', 'HOLLANDE': 'G', 'ROUSSEL': 'G',
    'MÉLENCHON': 'ExG', 'MELENCHON': 'ExG', 'POUTOU': 'ExG', 'ARTHAUD': 'ExG', 'JOLY': 'ExG'
}

df['Bord_Politique'] = df['Vainqueur'].map(lambda x: mapping_bord.get(x, 'Centre'))
print("Classes cibles identifiées :", df['Bord_Politique'].unique())
print(df['Bord_Politique'].value_counts())

# Feature Selection
feature_cols = ['Inscrits', 'Abstentions', 'Votants', 'Exprimés']
insee_deltas = [c for c in df.columns if 'delta_' in c][:10]
feature_cols += insee_deltas

for col in feature_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce').fillna(0)

X = df[feature_cols].copy()
y = df['Bord_Politique']

if len(y.unique()) > 1:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
    
    model = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    print(f"✅ Modèle entraîné (Accuracy: {accuracy_score(y, model.predict(X_scaled)):.2%})")
    
    # SAUVEGARDE
    models_dir = r'c:\Users\tarek\Downloads\aliMSPR\MSPR_Final\MSPR\03_Data_Science\models'
    if not os.path.exists(models_dir): os.makedirs(models_dir)
    with open(os.path.join(models_dir, 'model_nouvelle_aquitaine.pkl'), 'wb') as f: pickle.dump(model, f)
    with open(os.path.join(models_dir, 'scaler.pkl'), 'wb') as f: pickle.dump(scaler, f)
    with open(os.path.join(models_dir, 'features.pkl'), 'wb') as f: pickle.dump(feature_cols, f)
    
    df['Prediction_Bord'] = model.predict(X_scaled)
    df.to_csv(r'c:\Users\tarek\Downloads\aliMSPR\MSPR_Final\MSPR\01_Donnees\data_with_predictions.csv', index=False)
    print("✅ Fichiers sauvegardés avec succès.")
else:
    print("❌ Erreur : Pas assez de diversité dans 'Bord_Politique'.")

Dataset chargé : 127260 lignes, 33 colonnes
Étape 1 : Extraction du vainqueur...
Classes cibles identifiées : ['G' 'D' 'ExD' 'Centre' 'ExG']
Bord_Politique
G         69431
Centre    26040
D         15321
ExD       10924
ExG        5544
Name: count, dtype: int64
✅ Modèle entraîné (Accuracy: 100.00%)
✅ Fichiers sauvegardés avec succès.


In [ ]:
import pickle
import os

# Sauvegarde du modèle et du scaler pour l'application Flask
models_dir = r'c:\Users\tarek\Downloads\aliMSPR\MSPR_Final\MSPR\03_Data_Science\models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

model_path = os.path.join(models_dir, 'model_nouvelle_aquitaine.pkl')
scaler_path = os.path.join(models_dir, 'scaler.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

# Sauvegarde des colonnes de features utilisées
with open(os.path.join(models_dir, 'features.pkl'), 'wb') as f:
    pickle.dump(feature_cols, f)

# Sauvegarde des données avec prédictions pour l'affichage rapide (optionnel mais utile)
output_data_path = r'c:\Users\tarek\Downloads\aliMSPR\MSPR_Final\MSPR\01_Donnees\data_with_predictions.csv'
df.to_csv(output_data_path, index=False)

print(f"✅ Modèle sauvegardé dans : {model_path}")
print(f"✅ Scaler sauvegardé dans : {scaler_path}")
print(f"✅ Features sauvegardées dans : {os.path.join(models_dir, 'features.pkl')}")
print(f"✅ Données consolidées sauvegardées dans : {output_data_path}")